<a href="https://colab.research.google.com/github/Abhinavapsomayaji-AI/FIFA-World-cup-2026-Predictior-/blob/main/FIFA%202026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FIFA 2026 Predictor


In [ ]:
import pandas as pd

matches = pd.read_csv("results.csv")

rankings = pd.read_csv("fifa_mens_rank.csv")

print("HISTORICAL MATCHES")
print(matches.head())

print("\n FIFA RANKINGS")
print(rankings.head())

HISTORICAL MATCHES
         date home_team away_team  home_score  away_score tournament     city  \
0  1872-11-30  Scotland   England         0.0         0.0   Friendly  Glasgow   
1  1873-03-08   England  Scotland         4.0         2.0   Friendly   London   
2  1874-03-07  Scotland   England         2.0         1.0   Friendly  Glasgow   
3  1875-03-06   England  Scotland         2.0         2.0   Friendly   London   
4  1876-03-04  Scotland   England         3.0         0.0   Friendly  Glasgow   

    country  neutral  
0  Scotland    False  
1   England    False  
2  Scotland    False  
3   England    False  
4  Scotland    False  

 FIFA RANKINGS
   date  semester  rank       team acronym  total.points  previous.points  \
0  2024         2     1  Argentina     ARG       1867.25          1883.50   
1  2024         2     2     France     FRA       1859.78          1859.85   
2  2024         2     3      Spain     ESP       1853.27          1844.33   
3  2024         2     4    Engla

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving fifa_mens_rank.csv to fifa_mens_rank.csv
Saving former_names.csv to former_names.csv
Saving goalscorers.csv to goalscorers.csv
Saving results.csv to results.csv
Saving shootouts.csv to shootouts.csv
Saving wc_2026_fixtures.csv to wc_2026_fixtures.csv
Saving wc_2026_teams.csv to wc_2026_teams.csv
Saving wc_all_editions.csv to wc_all_editions.csv
Saving wc_all_matches.csv to wc_all_matches.csv
Saving wc_top_scorers.csv to wc_top_scorers.csv


In [ ]:
matches['date'] = pd.to_datetime(matches['date'])
modern_matches = matches[matches['date'].dt.year >= 2000].copy()

def determine_result(row):
    if row['home_score'] > row['away_score']:
        return 2
    elif row['home_score'] < row['away_score']:
        return 0
    else:
        return 1


modern_matches['match_result'] = modern_matches.apply(determine_result, axis=1)


print(f"Total modern matches to train on: {len(modern_matches)}")
print("\nLook at your new match_result column (2=Home Win, 1=Draw, 0=Away Win):")
print(modern_matches[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'match_result']].head())

Total modern matches to train on: 25383

Look at your new match_result column (2=Home Win, 1=Draw, 0=Away Win):
            date            home_team away_team  home_score  away_score  \
24062 2000-01-04                Egypt      Togo         2.0         1.0   
24063 2000-01-07              Tunisia      Togo         7.0         0.0   
24064 2000-01-08  Trinidad and Tobago    Canada         0.0         0.0   
24065 2000-01-09         Burkina Faso     Gabon         1.0         1.0   
24066 2000-01-09            Guatemala   Armenia         1.0         1.0   

       match_result  
24062             2  
24063             2  
24064             1  
24065             1  
24066             1  


In [ ]:
print(rankings.columns)

Index(['date', 'semester', 'rank', 'team', 'acronym', 'total.points',
       'previous.points', 'diff.points'],
      dtype='object')


In [ ]:
print("Rankings Columns: ",rankings.columns )
print(rankings.head(2))

Rankings Columns:  Index(['date', 'semester', 'rank', 'team', 'acronym', 'total.points',
       'previous.points', 'diff.points'],
      dtype='object')
   date  semester  rank       team acronym  total.points  previous.points  \
0  2024         2     1  Argentina     ARG       1867.25          1883.50   
1  2024         2     2     France     FRA       1859.78          1859.85   

   diff.points  
0       -16.25  
1        -0.07  


In [ ]:
latest_rankings = rankings.sort_values('date').groupby('team').last().reset_index()

matches_with_home_rank = pd.merge(
    modern_matches,
    latest_rankings[["team", "rank"]],
    left_on='home_team',
    right_on='team',
    how='left')

matches_with_home_rank = matches_with_home_rank.rename(columns={'rank': 'home_rank'})

final_dataset = pd.merge(
    matches_with_home_rank,
    latest_rankings[['team', 'rank']],
    left_on='away_team',
    right_on='team',
    how='left')

final_dataset = final_dataset.rename(columns={'rank': 'away_rank'})

final_dataset = final_dataset.dropna(subset=['home_rank', 'away_rank'])

final_dataset['rank_difference'] = final_dataset['home_rank'] - final_dataset['away_rank']


print(final_dataset[['date', 'home_team', 'away_team', 'home_rank', 'away_rank', 'rank_difference', 'match_result']].head())

        date            home_team away_team  home_rank  away_rank  \
0 2000-01-04                Egypt      Togo       33.0      119.0   
1 2000-01-07              Tunisia      Togo       52.0      119.0   
2 2000-01-08  Trinidad and Tobago    Canada      102.0       31.0   
3 2000-01-09         Burkina Faso     Gabon       66.0       84.0   
4 2000-01-09            Guatemala   Armenia      105.0      100.0   

   rank_difference  match_result  
0            -86.0             2  
1            -67.0             2  
2             71.0             1  
3            -18.0             1  
4              5.0             1  


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X = final_dataset[['rank_difference']]
y = final_dataset['match_result']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)

model.fit(X_train, y_train)

predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"Model Accuracy: {accuracy * 100:.2f}%")

Model Accuracy: 57.50%


In [ ]:
fixtures = pd.read_csv("wc_2026_fixtures.csv")

fixtures_with_home = pd.merge(fixtures, latest_rankings[['team', 'rank']], left_on='team1', right_on='team', how='left').rename(columns={'rank': 'home_rank'})
final_fixtures = pd.merge(fixtures_with_home, latest_rankings[['team', 'rank']], left_on='team2', right_on='team', how='left').rename(columns={'rank': 'away_rank'})

final_fixtures['home_rank'] = final_fixtures['home_rank'].fillna(50)
final_fixtures['away_rank'] = final_fixtures['away_rank'].fillna(50)

final_fixtures['rank_difference'] = final_fixtures['home_rank'] - final_fixtures['away_rank']

fixture_predictions = model.predict(final_fixtures[['rank_difference']])
final_fixtures['predicted_result_code'] = fixture_predictions

def code_to_text(code):
    if code == 2: return "Home Team Win"
    elif code == 0: return "Away Team Win"
    else: return "Draw"

final_fixtures['AI_Prediction'] = final_fixtures['predicted_result_code'].apply(code_to_text)

print(final_fixtures[['team1', 'team2', 'AI_Prediction']])

           team1         team2  AI_Prediction
0         Mexico  South Africa  Home Team Win
1    South Korea       Czechia  Home Team Win
2    South Korea        Mexico  Home Team Win
3        Czechia  South Africa  Home Team Win
4        Czechia        Mexico  Away Team Win
..           ...           ...            ...
99           QF7           QF8  Home Team Win
100          SF1           SF2  Home Team Win
101          SF3           SF4  Home Team Win
102    3rd Place     3rd Place  Home Team Win
103   Finalist 1    Finalist 2  Home Team Win

[104 rows x 3 columns]
